<a href="https://colab.research.google.com/github/rajshree23d/ecommerce-data-analysis/blob/main/Data_profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Olist E-Commerce Data Analysis

## Objective
Analyze e-commerce sales, customer behavior, product performance,
delivery performance, and customer satisfaction using Python.

## Tools
- Python
- Pandas
- NumPy
- Matplotlib
- SQL
- Power BI

## Dataset
Brazilian E-Commerce Public Dataset by Olist

In [1]:
# ============================================
# OLIST E-COMMERCE ANALYTICS
# Advanced Data Analytics Portfolio Project
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from datetime import datetime
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries loaded successfully")
print(f"📅 Analysis Date: {datetime.now().strftime('%Y-%m-%d')}")

data_path = "/content/drive/MyDrive/Colab Notebooks/Data_Analytics_portfolio/Olist_Ecommerce_Analytics"
os.listdir(data_path)
orders = pd.read_csv(
    f"{data_path}/olist_orders_dataset.csv"
)


# Load all CSVs
files = {
    'orders':        'olist_orders_dataset.csv',
    'order_items':   'olist_order_items_dataset.csv',
    'customers':     'olist_customers_dataset.csv',
    'products':      'olist_products_dataset.csv',
    'sellers':       'olist_sellers_dataset.csv',
    'payments':      'olist_order_payments_dataset.csv',
    'reviews':       'olist_order_reviews_dataset.csv',
    'category':      'product_category_name_translation.csv',
    'geolocation':   'olist_geolocation_dataset.csv'
}

dfs = {}
for name, file in files.items():
    path = os.path.join(data_path, file)
    dfs[name] = pd.read_csv(path)
    print(f"✅ {name:15} → {dfs[name].shape[0]:>7,} rows | {dfs[name].shape[1]:>2} cols")

# Unpack for easy access
orders       = dfs['orders']
order_items  = dfs['order_items']
customers    = dfs['customers']
products     = dfs['products']
sellers      = dfs['sellers']
payments     = dfs['payments']
reviews      = dfs['reviews']
category     = dfs['category']



✅ Libraries loaded successfully
📅 Analysis Date: 2026-09-15
✅ orders          →  99,441 rows |  8 cols
✅ order_items     → 112,650 rows |  7 cols
✅ customers       →  99,441 rows |  5 cols
✅ products        →  32,951 rows |  9 cols
✅ sellers         →   3,095 rows |  4 cols
✅ payments        → 103,886 rows |  5 cols
✅ reviews         →  99,224 rows |  7 cols
✅ category        →      71 rows |  2 cols
✅ geolocation     → 1,000,163 rows |  5 cols


In [2]:
# ============================================
# 2. DATA PROFILING — AUTOMATED REPORT
# ============================================

def profile_dataframe(df, name):
    """
    Generates a comprehensive data profile
    for any dataframe
    """
    print(f"\n{'='*60}")
    print(f" PROFILING: {name.upper()}")
    print(f"{'='*60}")
    print(f"  Shape:       {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Memory:      {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"  Duplicates:  {df.duplicated().sum():,}")

# Column-level profile
    profile = pd.DataFrame({
        'dtype':        df.dtypes,
        'null_count':   df.isnull().sum(),
        'null_pct':     (df.isnull().sum() / len(df) * 100).round(2),
        'unique_count': df.nunique(),
        'unique_pct':   (df.nunique() / len(df) * 100).round(2)
    })

# Add stats for numeric columns
    numeric_cols = df.select_dtypes(include=np.number).columns
    if len(numeric_cols) > 0:
        stats_df = df[numeric_cols].agg(['min','max','mean','median','std']).T
        profile = profile.join(stats_df, how='left')

    print(f"\n{profile.to_string()}")
    return profile

# Profile every dataset
profiles = {}
for name, df in dfs.items():
    if name != 'geolocation':  # skip large geo file
        profiles[name] = profile_dataframe(df, name)

print("\n✅ Data profiling complete")


 PROFILING: ORDERS
  Shape:       99,441 rows × 8 columns
  Memory:      52.94 MB
  Duplicates:  0

                                dtype  null_count  null_pct  unique_count  unique_pct
order_id                       object           0      0.00         99441      100.00
customer_id                    object           0      0.00         99441      100.00
order_status                   object           0      0.00             8        0.01
order_purchase_timestamp       object           0      0.00         98875       99.43
order_approved_at              object         160      0.16         90733       91.24
order_delivered_carrier_date   object        1783      1.79         81018       81.47
order_delivered_customer_date  object        2965      2.98         95664       96.20
order_estimated_delivery_date  object           0      0.00           459        0.46

 PROFILING: ORDER_ITEMS
  Shape:       112,650 rows × 7 columns
  Memory:      35.99 MB
  Duplicates:  0

                  

In [3]:
# ============================================
# 3. DATA CLEANING
# ============================================

print("🧹 Starting data cleaning...\n")

# ---- ORDERS ----
orders_clean = orders.copy()

# Convert all date columns
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders_clean[col] = pd.to_datetime(
        orders_clean[col], errors='coerce'
    )

    # Filter delivered orders only
orders_clean = orders_clean[
    orders_clean['order_status'] == 'delivered'
].copy()

# Remove nulls on critical delivery date
orders_clean = orders_clean.dropna(
    subset=['order_delivered_customer_date']
)

# Engineer date features
orders_clean['purchase_year']    = orders_clean['order_purchase_timestamp'].dt.year
orders_clean['purchase_month']   = orders_clean['order_purchase_timestamp'].dt.month
orders_clean['purchase_quarter'] = orders_clean['order_purchase_timestamp'].dt.quarter
orders_clean['purchase_dow']     = orders_clean['order_purchase_timestamp'].dt.dayofweek
orders_clean['purchase_hour']    = orders_clean['order_purchase_timestamp'].dt.hour

# Delivery metrics
orders_clean['delivery_days'] = (
    orders_clean['order_delivered_customer_date'] -
    orders_clean['order_purchase_timestamp']
).dt.days

orders_clean['approval_days'] = (
    orders_clean['order_approved_at'] -
    orders_clean['order_purchase_timestamp']
).dt.days

orders_clean['estimated_days'] = (
    orders_clean['order_estimated_delivery_date'] -
    orders_clean['order_purchase_timestamp']
).dt.days

orders_clean['on_time']         = (
    orders_clean['order_delivered_customer_date'] <=
    orders_clean['order_estimated_delivery_date']
)

orders_clean['early_late_days'] = (
    orders_clean['order_estimated_delivery_date'] -
    orders_clean['order_delivered_customer_date']
).dt.days  # positive = early, negative = late

# Remove delivery outliers
q_low  = orders_clean['delivery_days'].quantile(0.01)
q_high = orders_clean['delivery_days'].quantile(0.99)
orders_clean = orders_clean[
    orders_clean['delivery_days'].between(q_low, q_high)
]

print(f"✅ Orders cleaned: {len(orders_clean):,} delivered orders")

# ---- PRODUCTS ----
products_clean = products.copy()
products_clean['product_category_name'] = (
    products_clean['product_category_name'].fillna('unknown')
)
# Fill missing dimensions with median
dim_cols = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]
for col in dim_cols:
    products_clean[col] = products_clean[col].fillna(
        products_clean[col].median()
    )

    # Merge English category names
products_clean = products_clean.merge(
    category,
    on='product_category_name',
    how='left'
)
products_clean['category_en'] = (
    products_clean['product_category_name_english']
    .fillna(products_clean['product_category_name'])
)

print(f"✅ Products cleaned: {len(products_clean):,} products")

# ---- REVIEWS ----
reviews_clean = reviews.copy()
reviews_clean['review_creation_date'] = pd.to_datetime(
    reviews_clean['review_creation_date'], errors='coerce'
)
reviews_clean['review_comment_length'] = (
    reviews_clean['review_comment_message']
    .fillna('')
    .apply(len)
)
reviews_clean['has_comment'] = (
    reviews_clean['review_comment_message'].notna()
)

print(f"✅ Reviews cleaned: {len(reviews_clean):,} reviews")

# ---- PAYMENTS ----
payments_clean = payments.copy()
# Remove zero value payments
payments_clean = payments_clean[
    payments_clean['payment_value'] > 0
]

# Aggregate by order
payments_agg = payments_clean.groupby('order_id').agg(
    total_payment    = ('payment_value', 'sum'),
    payment_type     = ('payment_type', 'first'),
    installments     = ('payment_installments', 'max'),
    payment_count    = ('payment_sequential', 'count')
).reset_index()

print(f"✅ Payments aggregated: {len(payments_agg):,} orders")


🧹 Starting data cleaning...

✅ Orders cleaned: 95,577 delivered orders
✅ Products cleaned: 32,951 products
✅ Reviews cleaned: 99,224 reviews
✅ Payments aggregated: 99,437 orders


In [4]:
# ============================================
# 4. BUILD MASTER ANALYTICS TABLE
# ============================================

print("🔗 Building master analytics table...")

master = (
    order_items
    .merge(orders_clean,    on='order_id',    how='inner')
    .merge(customers,       on='customer_id', how='left')
    .merge(products_clean,  on='product_id',  how='left')
    .merge(payments_agg,    on='order_id',    how='left')
    .merge(
        reviews_clean[['order_id','review_score',
                        'has_comment','review_comment_length']],
        on='order_id', how='left'
    )
    .merge(
        sellers[['seller_id','seller_state','seller_city']],
        on='seller_id', how='left'
    )
)

# Revenue calculation
master['item_revenue']   = master['price'] + master['freight_value']
master['freight_ratio']  = (
    master['freight_value'] /
    master['price'].replace(0, np.nan)
).round(4)

# Price segments
master['price_segment'] = pd.cut(
    master['price'],
    bins=[0, 50, 100, 250, 500, float('inf')],
    labels=['Budget','Economy','Mid-Range','Premium','Luxury']
)

print(f"✅ Master table: {master.shape[0]:,} rows × {master.shape[1]} columns")

# Save master
master.to_csv(
    os.path.join(data_path, 'master_analytics.csv'),
    index=False
)
print("✅ Master table saved")

🔗 Building master analytics table...
✅ Master table: 109,845 rows × 50 columns
✅ Master table saved
